In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings

#### 1. PERSIAPAN DATA (PREPROCESSING)

In [35]:
# ==========================================
# 1. PERSIAPAN DATA (PREPROCESSING)
# ==========================================

# 1. Memuat dataset
file_path = "(CSV) Dataset_Utama - Draft_Pick.csv"
df = pd.read_csv(file_path)
print(df.head())

# 2. Logika penentuan status kemenangan tim
def check_win(row):
    if row['Team_Side'] == 'Blue' and row['Match_Outcome'] == 'Blue_Win':
        return True
    elif row['Team_Side'] == 'Red' and row['Match_Outcome'] == 'Red_Win':
        return True
    return False

df['Is_Win'] = df.apply(check_win, axis=1)

# 3. Pisahkan data untuk picks dan bans
df_picks = df[df['Is_Ban'] == False].copy()
df_bans = df[df['Is_Ban'] == True].copy()
total_matches = df['Match_ID'].nunique()

# 4. Parameter Global Apriori (0.01% support threshold) 
MIN_SUPPORT = 0.0001 

# 5. Fungsi Bantuan: Visualisasi Distribusi Support
def get_distribution(support_series, title):
    bins = [0.0001, 0.0002, 0.0005, 0.001, 0.01, 0.1, 1.0]
    labels = [
        '0.01% < s <= 0.02%',
        '0.02% < s <= 0.05%',
        '0.05% < s <= 0.10%',
        '0.10% < s <= 1.00%',
        '1.00% < s <= 10.0%',
        '> 10.0%'
    ]

    dist_categories = pd.cut(support_series, bins=bins, labels=labels, right=True)
    dist_counts = dist_categories.value_counts().reindex(labels[::-1]).fillna(0).astype(int)
    
    df_dist = pd.DataFrame({
        'Support Range': dist_counts.index,
        'Total Rules': dist_counts.values
    })

    total_rules = dist_counts.sum()
    total_row = pd.DataFrame({
        'Support Range': ['TOTAL'],
        'Total Rules': [total_rules]
    })

    df_dist = pd.concat([df_dist, total_row], ignore_index=True)

    print(f"\n--- DISTRIBUSI: {title} ---")
    print(df_dist.to_string(index=False))

    return df_dist

   NO Tanggal_Match          Match_ID  Game_Rounds Team_Name Team_Side  \
0   1      5/2/2025  Match_20250502_1            1      NMSS      Blue   
1   2      5/2/2025  Match_20250502_1            1       DMT       Red   
2   3      5/2/2025  Match_20250502_1            1      NMSS      Blue   
3   4      5/2/2025  Match_20250502_1            1       DMT       Red   
4   5      5/2/2025  Match_20250502_1            1      NMSS      Blue   

   Draft_Order  Is_Ban  Hero_ID     Hero_Name Hero_Tipe   Hero_Role  \
0            1    True      134        Dharma   Fighter  Clash_Lane   
1            2    True      157  Mai_Shiranui   Assasin    Mid_Lane   
2            3    True      531          Jing   Assasin     Jungler   
3            4    True      146          Luna   Fighter     Jungler   
4            5   False      199          Arli  Marksman   Farm_Lane   

  Match_Outcome  
0       Red_Win  
1       Red_Win  
2       Red_Win  
3       Red_Win  
4       Red_Win  


# Association Rule Mining

#### 2. EKSTRAKSI ATURAN ASOSIASI DENGAN APRIORI

In [29]:
# ==========================================
# ATURAN 1 (ALLIES / BASIS KAWAN)
# ==========================================
basket_allies = []

# Ekstrak data Win/Lose per tim
for (match_id, team_name), group in df_picks.groupby(['Match_ID', 'Team_Name']):
    is_win = group['Is_Win'].iloc[0]
    heroes = group['Hero_Name'].tolist()
    if is_win: basket_allies.append([f"Win_{h}" for h in heroes])
    else: basket_allies.append([f"Lose_{h}" for h in heroes])

te_allies = TransactionEncoder()
df_basket_allies = pd.DataFrame(te_allies.fit(basket_allies).transform(basket_allies), columns=te_allies.columns_)

freq_allies = apriori(df_basket_allies, min_support=MIN_SUPPORT, use_colnames=True)
rules_ap_allies = association_rules(freq_allies, metric="support", min_threshold=MIN_SUPPORT)
rules_ap_allies = rules_ap_allies[(rules_ap_allies['antecedents'].apply(len) + rules_ap_allies['consequents'].apply(len)) <= 5]

# Filter aturan (Win -> Win)
def is_valid_allies(ant, con):
    return all(str(i).startswith('Win_') for i in ant) and all(str(i).startswith('Win_') for i in con)

rules_allies = rules_ap_allies[rules_ap_allies.apply(lambda row: is_valid_allies(row['antecedents'], row['consequents']), axis=1)].copy()

# Strict Win Rate Kalkulasi
support_dict_allies = pd.Series(freq_allies.support.values, index=freq_allies.itemsets).to_dict()

def calc_win_rate(row):
    full_lose = frozenset([i.replace('Win_', 'Lose_') for i in row['antecedents'].union(row['consequents'])])
    s_win, s_lose = row['support'], support_dict_allies.get(full_lose, 0.0)
    return s_win / (s_win + s_lose) if (s_win + s_lose) > 0 else 0.0

rules_allies['win_rate'] = rules_allies.apply(calc_win_rate, axis=1)
rules_allies = rules_allies.sort_values(by=['win_rate', 'support'], ascending=[False, False])

# Format Output
rules_allies['Team_Heroes'] = rules_allies['antecedents'].apply(lambda x: [i.replace('Win_', '') for i in x])
rules_allies['Rec_Ally'] = rules_allies['consequents'].apply(lambda x: [i.replace('Win_', '') for i in x])

print("TOP 5 ATURAN 1 (ALLIES)")
print(rules_allies[['Team_Heroes', 'Rec_Ally', 'support', 'win_rate']].head(5).to_string(index=False))
get_distribution(rules_allies['support'], "ATURAN 1 (ALLIES)")

TOP 5 ATURAN 1 (ALLIES)
   Team_Heroes       Rec_Ally  support  win_rate
       [Agudo]    [Di_Renjie] 0.028571       1.0
   [Di_Renjie]        [Agudo] 0.028571       1.0
[Mai_Shiranui]        [Chano] 0.019048       1.0
       [Chano] [Mai_Shiranui] 0.019048       1.0
   [Di_Renjie]       [Sun_Ce] 0.019048       1.0

--- DISTRIBUSI: ATURAN 1 (ALLIES) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          384
0.10% < s <= 1.00%        16382
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL        16766


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,384
2,0.10% < s <= 1.00%,16382
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,16766


In [30]:
# ==========================================
# ATURAN 2 (COUNTERS / BASIS LAWAN)
# ==========================================
basket_counters = []

# Ekstrak data Lose(Musuh) dan Win(Kita) per match
for match_id, match_data in df_picks.groupby('Match_ID'):
    heroes_menang = match_data[match_data['Is_Win'] == True]['Hero_Name'].tolist()
    heroes_kalah = match_data[match_data['Is_Win'] == False]['Hero_Name'].tolist()
    basket_counters.append([f"Lose_{h}" for h in heroes_kalah] + [f"Win_{h}" for h in heroes_menang])

te_counter = TransactionEncoder()
df_basket_counter = pd.DataFrame(te_counter.fit(basket_counters).transform(basket_counters), columns=te_counter.columns_)

freq_counters = apriori(df_basket_counter, min_support=MIN_SUPPORT, use_colnames=True)
rules_ap_counters = association_rules(freq_counters, metric="support", min_threshold=MIN_SUPPORT)

# Filter (1 Musuh -> 1 Counter)
def is_valid_counter(ant, con):
    if len(ant) != 1 or len(con) != 1: return False
    return all(str(i).startswith('Lose_') for i in ant) and all(str(i).startswith('Win_') for i in con)

rules_counters = rules_ap_counters[rules_ap_counters.apply(lambda r: is_valid_counter(r['antecedents'], r['consequents']), axis=1)].copy()

# Kalkulasi Counter Coefficient
support_dict_counters = pd.Series(freq_counters.support.values, index=freq_counters.itemsets).to_dict()

def calc_counter_coef(row):
    hero_e = list(row['antecedents'])[0].replace('Lose_', '')
    hero_r = list(row['consequents'])[0].replace('Win_', '')
    s_e_to_r = row['support']
    s_r_to_e = support_dict_counters.get(frozenset({f"Lose_{hero_r}", f"Win_{hero_e}"}), 0.0)
    return s_e_to_r / (s_e_to_r + s_r_to_e) if (s_e_to_r + s_r_to_e) > 0 else 0.0

rules_counters['counter_coef'] = rules_counters.apply(calc_counter_coef, axis=1)
rules_counters = rules_counters.sort_values(by=['confidence', 'counter_coef'], ascending=[False, False])

# Format Output
rules_counters['Enemy'] = rules_counters['antecedents'].apply(lambda x: list(x)[0].replace('Lose_', ''))
rules_counters['Rec_Counter'] = rules_counters['consequents'].apply(lambda x: list(x)[0].replace('Win_', ''))

print("TOP 5 ATURAN 2 (COUNTERS)")
print(rules_counters[['Enemy', 'Rec_Counter', 'support', 'confidence', 'counter_coef']].head(5).to_string(index=False))
get_distribution(rules_counters['support'], "ATURAN 2 (COUNTERS)")

TOP 5 ATURAN 2 (COUNTERS)
 Enemy Rec_Counter  support  confidence  counter_coef
Angela        Arli 0.009524         1.0           1.0
Angela         Dun 0.009524         1.0           1.0
Angela  Gan_And_Mo 0.009524         1.0           1.0
Angela      Sakeer 0.009524         1.0           1.0
Angela      Sun_Ce 0.009524         1.0           1.0

--- DISTRIBUSI: ATURAN 2 (COUNTERS) ---
     Support Range  Total Rules
           > 10.0%            1
1.00% < s <= 10.0%          620
0.10% < s <= 1.00%          776
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL         1397


,Support Range,Total Rules
0,> 10.0%,1
1,1.00% < s <= 10.0%,620
2,0.10% < s <= 1.00%,776
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,1397


In [31]:
# ==========================================
# CELL 4: ATURAN 3 (META / EFFECTIVE BANS)
# ==========================================
# Ambil daftar ban dari tim yang berhasil menang
basket_meta_ban = df_bans[df_bans['Is_Win'] == True].groupby(['Match_ID', 'Team_Name'])['Hero_Name'].apply(list).tolist()

te_meta = TransactionEncoder()
df_basket_meta = pd.DataFrame(te_meta.fit(basket_meta_ban).transform(basket_meta_ban), columns=te_meta.columns_)

freq_meta = apriori(df_basket_meta, min_support=MIN_SUPPORT, use_colnames=True)
rules_meta = association_rules(freq_meta, metric="support", min_threshold=MIN_SUPPORT)
rules_meta = rules_meta.sort_values(by='support', ascending=False)

# Format Output (Apriori biasa)
rules_meta['Target_Ban'] = rules_meta['consequents'].apply(lambda x: list(x))

print("TOP 5 ATURAN 3 (META BANS)")
print(rules_meta[['antecedents', 'Target_Ban', 'support', 'confidence']].head(5).to_string(index=False))
get_distribution(rules_meta['support'], "ATURAN 3 (META BANS)")

TOP 5 ATURAN 3 (META BANS)
   antecedents     Target_Ban  support  confidence
(Mai_Shiranui)         [Arli] 0.169811    0.375000
        (Arli) [Mai_Shiranui] 0.169811    0.529412
(Mai_Shiranui)       [Dharma] 0.113208    0.250000
        (Luna)       [Dharma] 0.113208    0.285714
      (Dharma)         [Luna] 0.113208    0.413793

--- DISTRIBUSI: ATURAN 3 (META BANS) ---
     Support Range  Total Rules
           > 10.0%            8
1.00% < s <= 10.0%          526
0.10% < s <= 1.00%         3706
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL         4240


,Support Range,Total Rules
0,> 10.0%,8
1,1.00% < s <= 10.0%,526
2,0.10% < s <= 1.00%,3706
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,4240


In [32]:
# ---------------------------------------------------------
# PERSIAPAN KERANJANG SEKUENSIAL (Combo Breaker & Anti-Counter)
# ---------------------------------------------------------
basket_combo = []
basket_anti = []

for match_id, match_data in df.groupby('Match_ID'):
    match_data = match_data.sort_values(by='Draft_Order')
    
    for team in ['Blue', 'Red']:
        enemy_team = 'Red' if team == 'Blue' else 'Blue'
        
        # Ekstrak Pick Fase 1
        our_picks = match_data[(match_data['Is_Ban']==False) & (match_data['Team_Side']==team)].head(3)['Hero_Name'].tolist()
        enemy_picks = match_data[(match_data['Is_Ban']==False) & (match_data['Team_Side']==enemy_team)].head(3)['Hero_Name'].tolist()
        
        # Ekstrak Ban Fase 2 (2 ban terakhir per tim)
        our_phase2_bans = match_data[(match_data['Is_Ban']==True) & (match_data['Team_Side']==team)].tail(2)['Hero_Name'].tolist()
        
        if our_phase2_bans:
            # Combo Breaker format (Pick Musuh -> Ban Kita)
            basket_combo.append([f"EnemyPick_{h}" for h in enemy_picks] + [f"OurBan_{h}" for h in our_phase2_bans])
            # Anti-Counter format (Pick Kita -> Ban Kita)
            basket_anti.append([f"OurPick_{h}" for h in our_picks] + [f"OurBan_{h}" for h in our_phase2_bans])

te_seq = TransactionEncoder()

# ---------------------------------------------------------
# ATURAN 4: COMBO BREAKER BAN
# ---------------------------------------------------------
df_basket_combo = pd.DataFrame(te_seq.fit(basket_combo).transform(basket_combo), columns=te_seq.columns_)
freq_combo = apriori(df_basket_combo, min_support=MIN_SUPPORT, use_colnames=True)
rules_combo = association_rules(freq_combo, metric="support", min_threshold=MIN_SUPPORT)

# Filter: Antecedents = EnemyPick, Consequents = OurBan
rules_combo = rules_combo[
    rules_combo['antecedents'].apply(lambda x: all(str(i).startswith('EnemyPick_') for i in x)) &
    rules_combo['consequents'].apply(lambda x: all(str(i).startswith('OurBan_') for i in x))
]


print("\n--- TOP 5 ATURAN COMBO BREAKER (PICK MUSUH -> BAN KITA) ---")
print(rules_combo[['antecedents', 'consequents', 'support', 'confidence']].head(5))
get_distribution(rules_combo['support'], "ATURAN 4 (COMBO BREAKER)")


--- TOP 5 ATURAN COMBO BREAKER (PICK MUSUH -> BAN KITA) ---
          antecedents         consequents   support  confidence
21  (EnemyPick_Agudo)      (OurBan_Chano)  0.004762    0.090909
23  (EnemyPick_Agudo)  (OurBan_Charlotte)  0.004762    0.090909
25  (EnemyPick_Agudo)     (OurBan_Dharma)  0.004762    0.090909
27  (EnemyPick_Agudo)      (OurBan_Dolia)  0.004762    0.090909
29  (EnemyPick_Agudo)     (OurBan_Dyadia)  0.004762    0.090909

--- DISTRIBUSI: ATURAN 4 (COMBO BREAKER) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          160
0.10% < s <= 1.00%         3466
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL         3626


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,160
2,0.10% < s <= 1.00%,3466
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,3626


In [33]:
# ---------------------------------------------------------
# ATURAN 5: ANTI-COUNTER BAN
# ---------------------------------------------------------
df_basket_anti = pd.DataFrame(te_seq.fit(basket_anti).transform(basket_anti), columns=te_seq.columns_)
freq_anti = apriori(df_basket_anti, min_support=MIN_SUPPORT, use_colnames=True)
rules_anti = association_rules(freq_anti, metric="support", min_threshold=MIN_SUPPORT)

# Filter: Antecedents = OurPick, Consequents = OurBan
rules_anti = rules_anti[
    rules_anti['antecedents'].apply(lambda x: all(str(i).startswith('OurPick_') for i in x)) &
    rules_anti['consequents'].apply(lambda x: all(str(i).startswith('OurBan_') for i in x))
]

print("\n--- TOP 5 ATURAN ANTI-COUNTER (PICK KITA -> BAN KITA) ---")
print(rules_anti[['antecedents', 'consequents', 'support', 'confidence']].head(5))
get_distribution(rules_anti['support'], "ATURAN 5 (ANTI-COUNTER)")


--- TOP 5 ATURAN ANTI-COUNTER (PICK KITA -> BAN KITA) ---
         antecedents     consequents   support  confidence
27  (OurPick_Allain)  (OurBan_Agudo)  0.009524    0.142857
29  (OurPick_Ao_Yin)  (OurBan_Agudo)  0.014286    0.066667
31    (OurPick_Arli)  (OurBan_Agudo)  0.019048    0.105263
33  (OurPick_Augran)  (OurBan_Agudo)  0.009524    0.181818
35   (OurPick_Biron)  (OurBan_Agudo)  0.004762    0.035714

--- DISTRIBUSI: ATURAN 5 (ANTI-COUNTER) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          153
0.10% < s <= 1.00%         3585
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL         3738


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,153
2,0.10% < s <= 1.00%,3585
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,3738


In [34]:
# ==========================================
# RINGKASAN TOTAL 5 ATURAN
# ==========================================
total_allies = len(rules_allies)
total_counters = len(rules_counters)
total_meta = len(rules_meta)
total_combo = len(rules_combo)
total_anti = len(rules_anti)
total_keseluruhan = total_allies + total_counters + total_meta + total_combo + total_anti

print("==========================================")
print("     📊 RINGKASAN DATASET & ATURAN 📊     ")
print("==========================================")
print(f"Total Pertandingan : {df['Match_ID'].nunique()} Match")
print("------------------------------------------")
print(f"1. Aturan Basis Kawan (Allies)     : {total_allies} Rules")
print(f"2. Aturan Basis Lawan (Counters)   : {total_counters} Rules")
print(f"3. Aturan Meta Bans                : {total_meta} Rules")
print(f"4. Aturan Combo Breaker Bans       : {total_combo} Rules")
print(f"5. Aturan Anti-Counter Bans        : {total_anti} Rules")
print("==========================================")
print(f"TOTAL ATURAN TEREKSTRAKSI          : {total_keseluruhan} Rules")
print("==========================================")

     📊 RINGKASAN DATASET & ATURAN 📊     
Total Pertandingan : 105 Match
------------------------------------------
1. Aturan Basis Kawan (Allies)     : 16766 Rules
2. Aturan Basis Lawan (Counters)   : 1397 Rules
3. Aturan Meta Bans                : 4240 Rules
4. Aturan Combo Breaker Bans       : 3626 Rules
5. Aturan Anti-Counter Bans        : 3738 Rules
TOTAL ATURAN TEREKSTRAKSI          : 29767 Rules
